<a href="https://colab.research.google.com/github/camlet0630/BERT_with_FHE/blob/colab/BERT_with_FHE_20Newsgroups.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

說明：這部分如果出現需要重新啟動的報錯 (You must restart the runtime in order to use newly installed versions.)，重新啟動就可以了(這個 cell 可能需要跑兩次)

In [ ]:
! pip install -U pip setuptools
! pip install concrete-ml transformers datasets

## Prepare

In [ ]:
! pip install transformers

In [ ]:
import pandas as pd
import numpy as np
import os
import gc
from tqdm import tqdm
import random
from collections import Counter
from IPython.display import display


import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizerFast, AutoModel
from transformers import AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold

In [ ]:
device = torch.device('cuda', 0) if torch.cuda.is_available() else 'cpu'
print(device)

# BERT Tokenizer: 使用 'bert-base-cased' 版本
tokenizer = BertTokenizerFast.from_pretrained('bert-base-cased')

# BERT Model: 使用 'bert-base-cased' 版本
bert = AutoModel.from_pretrained('bert-base-cased').to(device)

In [ ]:
# config
config = {"embedding_path": "embedding_tensor.pt",
          "seed": 168,
          "learning_rate": 3e-4,
          "batch_size": 64,
          "epochs": 300}

In [ ]:
def set_random_seed(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)

set_random_seed(config["seed"])

## Data preparation

In [ ]:
from numpy.lib.shape_base import apply_along_axis
# fetch the dataset using scikit-learn
import sklearn
from sklearn.datasets import fetch_20newsgroups
train_b = fetch_20newsgroups(subset='train', shuffle=True, random_state=42)
test_b = fetch_20newsgroups(subset='test', shuffle=True, random_state=42)
all_data = fetch_20newsgroups(subset='all', shuffle=True, random_state=42)

print('size of training set: %s' % (len(train_b['data'])))
print('size of validation set: %s' % (len(test_b['data'])))
print('classes: %s' % (train_b.target_names))

x_train = train_b.data
y_train = train_b.target
x_test = test_b.data
y_test = test_b.target

In [ ]:
print(y_train)

## BERT Embedding

In [ ]:
def get_embedding_tensor(context_ary):

    embedding_tensor = torch.ones((context_ary.shape[0], 768))


    for i in range(context_ary.shape[0]):

        context = context_ary[i]
        bert_input = tokenizer(context, padding='max_length', max_length=512,
                               truncation=True, return_tensors="pt")


        attention_mask = bert_input['attention_mask'].to(device)
        input_id = bert_input['input_ids'].squeeze(1).to(device)
        final_inputs = {'input_ids': input_id, 'attention_mask': attention_mask}
        outputs = bert(**final_inputs)

        ## Only the embedding vector of [CLS] is taken
        pooler_output = outputs.last_hidden_state[0][0].reshape(768)
        embedding_tensor[i] = pooler_output.detach().cpu()

        gc.collect()
        torch.cuda.empty_cache()

    return embedding_tensor

In [ ]:
# 計算每個樣本各自的 BERT Embedding
train_context_ary = np.array(x_train)
train_embedding = get_embedding_tensor(train_context_ary)

test_context_ary = np.array(x_test)
test_embedding = get_embedding_tensor(test_context_ary)

## Logistic Regression

follow by https://docs.zama.ai/concrete-ml/advanced-topics/prediction_with_fhe


In [ ]:
## https://docs.zama.ai/concrete-ml/advanced-topics/prediction_with_fhe
from concrete.ml.sklearn import LogisticRegression
import time

# Instantiate and train the model
model = LogisticRegression()
model.fit(train_embedding, y_train)

# Simulate the predictions in the clear
start = time.perf_counter()
y_pred_clear = model.predict(test_embedding)
end = time.perf_counter()
print(f"LR computing time: {end - start:.4f} seconds")

# Compile the model on a representative set
start = time.perf_counter()
fhe_circuit = model.compile(train_embedding)
end = time.perf_counter()
print(f"Compilation time: {end - start:.4f} seconds")

# Predict in FHE
start = time.perf_counter()
y_pred_fhe = model.predict(test_embedding, fhe="execute")
end = time.perf_counter()
fhe_exec_time = end - start
print(f"FHE inference time: {fhe_exec_time:.4f} seconds")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, classification_report

# accuracy_score
print(f"accuracy_score: {accuracy_score(y_test, y_pred_clear)}")

# precision, recall, f1-score
print(f"Micro precision: {precision_score(y_test, y_pred_clear, average='micro')}")
print(f"Micro recall: {recall_score(y_test, y_pred_clear, average='micro')}")
print(f"Micro f1-score: {f1_score(y_test, y_pred_clear, average='micro')}")

print(f"Macro precision: {precision_score(y_test, y_pred_clear, average='macro')}")
print(f"Macro recall: {recall_score(y_test, y_pred_clear, average='macro')}")
print(f"Macro f1-score: {f1_score(y_test, y_pred_clear, average='macro')}")


# FHE
print()
print("FHE:")
print(f"accuracy_score FHE: {accuracy_score(y_test, y_pred_fhe)}")
print(f"Micro precision FHE: {precision_score(y_test, y_pred_fhe, average='micro')}")
print(f"Micro recall FHE: {recall_score(y_test, y_pred_fhe, average='micro')}")
print(f"Micro f1-score FHE: {f1_score(y_test, y_pred_fhe, average='micro')}")

print(f"Macro precision FHE: {precision_score(y_test, y_pred_fhe, average='macro')}")
print(f"Macro recall FHE: {recall_score(y_test, y_pred_fhe, average='macro')}")
print(f"Macro f1-score FHE: {f1_score(y_test, y_pred_fhe, average='macro')}")


# difference between with or not FHE
print()
print("Loss")
print(f"accuracy_score Loss: {accuracy_score(y_test, y_pred_clear) - accuracy_score(y_test, y_pred_fhe)}")
print(f"Micro precision Loss: {precision_score(y_test, y_pred_clear, average='micro') - precision_score(y_test, y_pred_fhe, average='micro')}")
print(f"Micro recall Loss: {recall_score(y_test, y_pred_clear, average='micro') - recall_score(y_test, y_pred_fhe, average='micro')}")
print(f"Micro f1-score Loss: {f1_score(y_test, y_pred_clear, average='micro') - f1_score(y_test, y_pred_fhe, average='micro')}")
print(f"Macro precision Loss: {precision_score(y_test, y_pred_clear, average='macro') - precision_score(y_test, y_pred_fhe, average='macro')}")
print(f"Macro recall Loss: {recall_score(y_test, y_pred_clear, average='macro') - recall_score(y_test, y_pred_fhe, average='macro')}")
print(f"Macro f1-score Loss: {f1_score(y_test, y_pred_clear, average='macro') - f1_score(y_test, y_pred_fhe, average='macro')}")

# By XGBoost

程式運作時間過長

## Classifying with XGBoost

In [ ]:
"""
from concrete.ml.sklearn import XGBClassifier
from sklearn.model_selection import GridSearchCV
# Let's build our model
model = XGBClassifier()
model.fit(train_embedding_tensor, y_train)
# A gridsearch to find the best parameters
parameters = {
    "n_bits": [2, 3],
    "max_depth": [1],
    "n_estimators": [10, 30, 50],
    "n_jobs": [-1],
}

# Now we have a representation for each tweet, we can train a model on these.
grid_search = GridSearchCV(model, parameters, cv=5, n_jobs=5, scoring="accuracy")
grid_search.fit(train_embedding_tensor, y_train)

# Check the accuracy of the best model
print(f"Best score: {grid_search.best_score_}")

# Check best hyperparameters
print(f"Best parameters: {grid_search.best_params_}")

# Extract best model
best_model = grid_search.best_estimator_
"""

## Predicting over encrypted data

In [ ]:
"""
import time
# Compute the metrics on the test set
y_pred = best_model.predict(test_embedding_tensor)

# Compile the model to get the FHE inference engine
# (this may take a few minutes depending on the selected model)
start = time.perf_counter()
best_model.compile(train_embedding_tensor)
end = time.perf_counter()
print(f"Compilation time: {end - start:.4f} seconds")

# Now let's predict with FHE over a single tweet and print the time it takes
start = time.perf_counter()
decrypted_predic = best_model.predict(test_embedding_tensor, fhe="execute")
end = time.perf_counter()
fhe_exec_time = end - start
print(f"FHE inference time: {fhe_exec_time:.4f} seconds")

y_pred_HE = decrypted_predic
"""

In [ ]:
"""
# accuracy_score
print(f"accuracy_score: {accuracy_score(y_test, y_pred_clear)}")

# precision, recall, f1-score
print(f"Micro precision: {precision_score(y_test, y_pred_clear, average='micro')}")
print(f"Micro recall: {recall_score(y_test, y_pred_clear, average='micro')}")
print(f"Micro f1-score: {f1_score(y_test, y_pred_clear, average='micro')}")

print(f"Macro precision: {precision_score(y_test, y_pred_clear, average='macro')}")
print(f"Macro recall: {recall_score(y_test, y_pred_clear, average='macro')}")
print(f"Macro f1-score: {f1_score(y_test, y_pred_clear, average='macro')}")


# FHE
print()
print("FHE:")
print(f"accuracy_score FHE: {accuracy_score(y_test, y_pred_fhe)}")
print(f"Micro precision FHE: {precision_score(y_test, y_pred_fhe, average='micro')}")
print(f"Micro recall FHE: {recall_score(y_test, y_pred_fhe, average='micro')}")
print(f"Micro f1-score FHE: {f1_score(y_test, y_pred_fhe, average='micro')}")

print(f"Macro precision FHE: {precision_score(y_test, y_pred_fhe, average='macro')}")
print(f"Macro recall FHE: {recall_score(y_test, y_pred_fhe, average='macro')}")
print(f"Macro f1-score FHE: {f1_score(y_test, y_pred_fhe, average='macro')}")


# difference between with or not FHE
print()
print("Loss")
print(f"accuracy_score Loss: {accuracy_score(y_test, y_pred_clear) - accuracy_score(y_test, y_pred_fhe)}")
print(f"Micro precision Loss: {precision_score(y_test, y_pred_clear, average='micro') - precision_score(y_test, y_pred_fhe, average='micro')}")
print(f"Micro recall Loss: {recall_score(y_test, y_pred_clear, average='micro') - recall_score(y_test, y_pred_fhe, average='micro')}")
print(f"Micro f1-score Loss: {f1_score(y_test, y_pred_clear, average='micro') - f1_score(y_test, y_pred_fhe, average='micro')}")
print(f"Macro precision Loss: {precision_score(y_test, y_pred_clear, average='macro') - precision_score(y_test, y_pred_fhe, average='macro')}")
print(f"Macro recall Loss: {recall_score(y_test, y_pred_clear, average='macro') - recall_score(y_test, y_pred_fhe, average='macro')}")
print(f"Macro f1-score Loss: {f1_score(y_test, y_pred_clear, average='macro') - f1_score(y_test, y_pred_fhe, average='macro')}")
"""